In [ ]:
from google.colab import drive
import pandas as pd
import glob
import os
import re


In [ ]:
# 1. Google Drive'ı bağla
drive.mount('/content/drive')

# 2. Örnek CSV dosyası yolu (örneğin Yusuf Dikeç klasöründen ilk dosya)
path = "/content/drive/MyDrive/Twitter Verileri/Yusuf Dikeç/*.csv"
files = glob.glob(path)

# İlk dosyayı aç
sample_file = files[0]
print("Örnek dosya:", sample_file)

# 3. Dosyayı oku ve sütunları listele
df = pd.read_csv(sample_file)
print("Sütunlar:")
print(df.columns)

Mounted at /content/drive
Örnek dosya: /content/drive/MyDrive/Twitter Verileri/Yusuf Dikeç/yusufdikecenglish-106.csv
Sütunlar:
Index(['Css-175oi2r URL', 'Css-1jxf684', 'Css-1jxf684.1', 'Css-146c3p1 URL',
       'Time', 'Css-1jxf684.2', 'Css-175oi2r URL.1', 'Css-1jxf684.3',
       'Css-1jxf684.4', 'Css-1jxf684.5', 'Css-1jxf684.6', 'Css-1jxf684.7',
       'Css-1jxf684.8', 'Css-1jxf684.9', 'Css-175oi2r URL.2',
       'R-4qtqp9 Description'],
      dtype='object')


In [ ]:
# İlk 10 satırı gösterelim
for col in df.columns:
    print(f"\nSütun: {col}")
    print(df[col].head(10))



Sütun: Css-175oi2r URL
0    https://x.com/issf_official
1        https://x.com/hey_madni
2      https://x.com/ginacostag_
3      https://x.com/NBCOlympics
4     https://x.com/cyclingontnt
5        https://x.com/dw_sports
6        https://x.com/dw_sports
7     https://x.com/Ubermenscchh
8        https://x.com/komunisnt
9          https://x.com/veroz73
Name: Css-175oi2r URL, dtype: object

Sütun: Css-1jxf684
0                          ISSF
1                  Madni Aghadi
2                   Gina Acosta
3    NBC Olympics & Paralympics
4         Cycling on TNT Sports
5                     DW Sports
6                     DW Sports
7                         Hasan
8                    komunisn't
9                         veroz
Name: Css-1jxf684, dtype: object

Sütun: Css-1jxf684.1
0    @issf_official
1        @hey_madni
2      @ginacostag_
3      @NBCOlympics
4     @cyclingontnt
5        @dw_sports
6        @dw_sports
7     @Ubermenscchh
8        @komunisnt
9          @veroz73
Name: Css-1jxf

In [ ]:
base_path = "/content/drive/MyDrive/Twitter Verileri"

# Hedef klasörler
folders = {
    "Yusuf Dikeç": "Yusuf_Dikec_duzenli.csv",
    "Mete Gazoz": "Mete_Gazoz_duzenli.csv"
}

In [ ]:
import pandas as pd
import re
import os
from google.colab import drive

# Google Drive bağla
drive.mount('/content/drive')

# Ana klasör yolu
base_path = "/content/drive/MyDrive/Twitter Verileri"

# Sayısal değer temizleme (örn. 3.1K → 3100, 2M → 2000000)
def temizle_sayi_deger(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    x = x.replace(",", "").replace(".", ".")  # virgülleri temizle
    match = re.match(r"^([\d.]+)([kKmM]?)\s*([Vv]iews)?$", x)
    if not match:
        return None
    try:
        sayi = float(match.group(1))
    except:
        return None
    if match.group(2).lower() == "k":
        sayi *= 1000
    elif match.group(2).lower() == "m":
        sayi *= 1000000
    return sayi

# Sütunları dönüştüren fonksiyon
def extract_columns(df):
    result = pd.DataFrame(index=df.index)

    # hesapurl
    hesapurl_cols = [c for c in df.columns if "Css-175oi2r URL" in c and "." not in c]
    if hesapurl_cols:
        result["hesapurl"] = df[hesapurl_cols[0]]

    # hesapadi & hesapnickname
    hesap_cols = [c for c in df.columns if c.startswith("Css-1jxf684")]
    if len(hesap_cols) >= 2:
        result["hesapadi"] = df[hesap_cols[0]]
        result["hesapnickname"] = df[hesap_cols[1]]

    # tweeturl
    tweeturl_cols = [c for c in df.columns if "Css-146c3p1 URL" in c]
    if tweeturl_cols:
        result["tweeturl"] = df[tweeturl_cols[0]]

    # time
    if "Time" in df.columns:
        result["time"] = df["Time"]

    # emoji
    emoji_cols = [c for c in df.columns if "R-4qtqp9 Description" in c]
    if emoji_cols:
        result["emoji"] = df[emoji_cols].astype(str).apply(
            lambda x: " ".join([v for v in x if v != "nan"]), axis=1
        )

    # alintiurl
    alintiurl_cols = [c for c in df.columns if "Css-175oi2r URL." in c]
    if alintiurl_cols:
        result["alintiurl"] = df[alintiurl_cols].bfill(axis=1).iloc[:, 0]

    # sayısal sütunlar (yorum/retweet/beğeni)
    sayisal_cols = []
    for c in df.columns:
        if c.startswith("Css-1jxf684"):
            if df[c].dropna().astype(str).str.match(r"^\d+(\.\d+)?[kKmM]?\s*([Vv]iews)?$").all():
                sayisal_cols.append(c)

    if sayisal_cols:
        def ayir_sayilar(row):
            degerler = []
            for c in sayisal_cols:
                v_raw = str(row[c]) if pd.notna(row[c]) else None
                v = temizle_sayi_deger(v_raw)
                if v is not None and v <= 300000:  # 300k sınırı
                    degerler.append(v)
            if not degerler:
                return pd.Series([None, None, None])

            degerler_sorted = sorted(degerler)
            yorum = degerler_sorted[0] if len(degerler_sorted) > 0 else None
            retweet = degerler_sorted[1] if len(degerler_sorted) > 1 else None
            begeni = degerler_sorted[2] if len(degerler_sorted) > 2 else None

            # Mantık: yorum < retweet < beğeni
            if yorum and retweet and retweet < yorum:
                retweet = yorum + 1
            if retweet and begeni and begeni < retweet:
                begeni = retweet + 1

            return pd.Series([yorum, retweet, begeni])

        sayilar_df = df.apply(ayir_sayilar, axis=1)
        sayilar_df.columns = ["yorum", "retweet", "begeni"]
        result = pd.concat([result, sayilar_df], axis=1)

    # tweet
    tweet_cols = [c for c in hesap_cols if c not in sayisal_cols]
    if tweet_cols:
        result["tweet"] = df[tweet_cols].astype(str).apply(
            lambda x: " ".join([v for v in x if v != "nan"]), axis=1
        )

    return result

# Klasör işleme fonksiyonu
def process_folder(input_folder, output_file):
    all_dfs = []
    for file in os.listdir(input_folder):
        if file.endswith(".csv"):
            path = os.path.join(input_folder, file)
            try:
                df = pd.read_csv(path)
                transformed = extract_columns(df)
                all_dfs.append(transformed)
            except Exception as e:
                print(f"Hata: {file} işlenemedi → {e}")
    if all_dfs:
        final_df = pd.concat(all_dfs, ignore_index=True)
        final_df.to_csv(output_file, index=False, encoding="utf-8-sig")
        print(f"✅ Kaydedildi: {output_file} ({len(final_df)} satır)")

# İşlenecek klasörler
folders = {
    "Yusuf Dikeç": "Yusuf_Dikec_Duzenlenmis.csv",
    "Mete Gazoz": "Mete_Gazoz_Duzenlenmis.csv"
}

for folder, output in folders.items():
    process_folder(os.path.join(base_path, folder), os.path.join(base_path, output))



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Kaydedildi: /content/drive/MyDrive/Twitter Verileri/Yusuf_Dikec_Duzenlenmis.csv (15199 satır)
✅ Kaydedildi: /content/drive/MyDrive/Twitter Verileri/Mete_Gazoz_Duzenlenmis.csv (10815 satır)


In [ ]:
import pandas as pd
import os

# Klasör yolu
base_path = "/content/drive/MyDrive/Twitter Verileri"

# Düzenlenmiş dosyaların isimleri
files = ["Yusuf_Dikec_Duzenlenmis.csv", "Mete_Gazoz_Duzenlenmis.csv"]

def remove_duplicates_by_tweeturl(file_path):
    df = pd.read_csv(file_path)

    print(f"📌 {os.path.basename(file_path)}")
    print(f"   Önce: {len(df)} satır")

    # tweeturl bazlı benzersizleştirme
    df_cleaned = df.drop_duplicates(subset=["tweeturl"], keep="first")

    print(f"   Sonra: {len(df_cleaned)} satır (tweeturl'e göre temizlendi)")

    # Üzerine kaydet
    df_cleaned.to_csv(file_path, index=False, encoding="utf-8-sig")
    print(f"✅ Güncellendi: {file_path}\n")

# İşlem uygula
for file in files:
    file_path = os.path.join(base_path, file)
    remove_duplicates_by_tweeturl(file_path)




📌 Yusuf_Dikec_Duzenlenmis.csv
   Önce: 15199 satır
   Sonra: 6776 satır (tweeturl'e göre temizlendi)
✅ Güncellendi: /content/drive/MyDrive/Twitter Verileri/Yusuf_Dikec_Duzenlenmis.csv

📌 Mete_Gazoz_Duzenlenmis.csv
   Önce: 10815 satır
   Sonra: 3199 satır (tweeturl'e göre temizlendi)
✅ Güncellendi: /content/drive/MyDrive/Twitter Verileri/Mete_Gazoz_Duzenlenmis.csv



In [ ]:
# Yusuf Dikeç verisi kontrol
yusuf_path = os.path.join(base_path, "Yusuf_Dikec_Duzenlenmis.csv")
yusuf_df = pd.read_csv(yusuf_path)
print("📌 Yusuf Dikeç verisi — Toplam satır:", len(yusuf_df))
display(yusuf_df.head())

# Mete Gazoz verisi kontrol
mete_path = os.path.join(base_path, "Mete_Gazoz_Duzenlenmis.csv")
mete_df = pd.read_csv(mete_path)
print("📌 Mete Gazoz verisi — Toplam satır:", len(mete_df))
display(mete_df.head())


📌 Yusuf Dikeç verisi — Toplam satır: 6776


,hesapurl,hesapadi,hesapnickname,tweeturl,time,emoji,alintiurl,yorum,retweet,begeni,tweet
0,https://x.com/issf_official,ISSF,@issf_official,https://x.com/issf_official/status/18200369590...,"Aug 4, 2024",👀,https://x.com/issf_official/status/18200369590...,41.0,727.0,3900.0,ISSF @issf_official We found the young Dikec Y...
1,https://x.com/hey_madni,Madni Aghadi,@hey_madni,https://x.com/hey_madni/status/182333168398019...,"Aug 13, 2024",NaN,https://x.com/hey_madni/status/182333168398019...,435.0,3100.0,51000.0,Madni Aghadi @hey_madni AI is getting out of h...
2,https://x.com/ginacostag_,Gina Acosta,@ginacostag_,https://x.com/ginacostag_/status/1822848851931...,"Aug 12, 2024",NaN,NaN,NaN,NaN,NaN,Gina Acosta @ginacostag_
3,https://x.com/NBCOlympics,NBC Olympics & Paralympics,@NBCOlympics,https://x.com/NBCOlympics/status/1822475719668...,"Aug 11, 2024",😤,https://x.com/NBCOlympics/status/1822475719668...,11.0,73.0,625.0,NBC Olympics & Paralympics @NBCOlympics Team G...
4,https://x.com/cyclingontnt,Cycling on TNT Sports,@cyclingontnt,https://x.com/cyclingontnt/status/182235887730...,"Aug 10, 2024",🇬🇧,https://x.com/cyclingontnt/status/182235887730...,61.0,1100.0,15000.0,Cycling on TNT Sports @cyclingontnt Team GB hi...


📌 Mete Gazoz verisi — Toplam satır: 3199


,hesapurl,hesapadi,hesapnickname,tweeturl,time,emoji,alintiurl,yorum,retweet,begeni,tweet
0,https://x.com/Olympics,The Olympic Games,@Olympics,https://x.com/Olympics/status/1421381248400994307,"Jul 31, 2021",NaN,https://x.com/Olympics/status/1421381248400994...,576.0,14000.0,78000.0,The Olympic Games @Olympics Turkey’s first eve...
1,https://x.com/Tokyo2020,#Tokyo2020,@Tokyo2020,https://x.com/Tokyo2020/status/142138268643056...,"Jul 31, 2021",🇹🇷,https://x.com/Tokyo2020/status/142138268643056...,125.0,2700.0,18000.0,#Tokyo2020 @Tokyo2020 Mete is the first archer...
2,https://x.com/worldarchery,World Archery,@worldarchery,https://x.com/worldarchery/status/142136486317...,"Jul 31, 2021",💪,https://x.com/worldarchery/status/142136486317...,10.0,134.0,822.0,World Archery @worldarchery Mete upsets the wo...
3,https://x.com/TRTWorldNow,TRT World Now,@TRTWorldNow,https://x.com/TRTWorldNow/status/1421381591633...,"Jul 31, 2021",NaN,https://x.com/TRTWorldNow/status/1421381591633...,7.0,101.0,722.0,TRT World Now @TRTWorldNow Mete wins Turkey's ...
4,https://x.com/TRTWorldNow,TRT World Now,@TRTWorldNow,https://x.com/TRTWorldNow/status/1421417857309...,"Jul 31, 2021",NaN,NaN,NaN,NaN,NaN,TRT World Now @TRTWorldNow


In [ ]:
# Yusuf Dikeç verisi sıralama
yusuf_sorted = yusuf_df.sort_values(by="begeni", ascending=False)
print("📌 Yusuf Dikeç — En çok görüntülenen ilk 10 tweet")
display(yusuf_sorted.head(10))

# Mete Gazoz verisi sıralama
mete_sorted = mete_df.sort_values(by="begeni", ascending=False)
print("📌 Mete Gazoz — En çok görüntülenen ilk 10 tweet")
display(mete_sorted.head(10))


📌 Yusuf Dikeç — En çok görüntülenen ilk 10 tweet


,hesapurl,hesapadi,hesapnickname,tweeturl,time,emoji,alintiurl,yorum,retweet,begeni,tweet
886,https://x.com/yusufdikec,Yusuf Dikec,@yusufdikec,https://x.com/yusufdikec/status/18201863670301...,"Aug 4, 2024",😎,https://x.com/yusufdikec/status/18201863670301...,3000.0,21000.0,295000.0,"Yusuf Dikec @yusufdikec Hi Elon, do you think ..."
2090,https://x.com/yusufdikec,Yusuf Dikec,@yusufdikec,https://x.com/yusufdikec/status/18205647374591...,"Aug 5, 2024",🇸🇪,https://x.com/yusufdikec/status/18205647374591...,483.0,6700.0,190000.0,Yusuf Dikec @yusufdikec Congratulations Duplantis
1195,https://x.com/historyinmemes,Historic Vids,@historyinmemes,https://x.com/historyinmemes/status/1819129025...,"Aug 2, 2024",NaN,https://x.com/historyinmemes/status/1819129025...,1200.0,13000.0,173000.0,"Historic Vids @historyinmemes Yusuf , a 51-yea..."
5651,https://x.com/ibrahimtncrr,ibrahim,@ibrahimtncrr,https://x.com/ibrahimtncrr/status/181877514201...,"Aug 1, 2024",👏,https://x.com/ibrahimtncrr/status/181877514201...,95.0,1200.0,93000.0,ibrahim @ibrahimtncrr #Paris2024 https://x.com...
5671,https://x.com/TMOK_Olimpiyat,TMOK | #TeamTürkiye,@TMOK_Olimpiyat,https://x.com/TMOK_Olimpiyat/status/1818717922...,"Jul 31, 2024",🇹🇷,https://x.com/TMOK_Olimpiyat/status/1818717922...,188.0,2600.0,88000.0,TMOK | #TeamTürkiye @TMOK_Olimpiyat #Paris2024...
566,https://x.com/yusufdikec,Yusuf Dikec,@yusufdikec,https://x.com/yusufdikec/status/18208746165139...,"Aug 6, 2024",🎯,https://x.com/yusufdikec/status/18208746165139...,409.0,2300.0,84000.0,Yusuf Dikec @yusufdikec Filenin Sultanları 12'...
411,https://x.com/cyclingontnt,Cycling on TNT Sports,@cyclingontnt,https://x.com/cyclingontnt/status/181897632134...,"Aug 1, 2024",😳,https://x.com/cyclingontnt/status/181897632134...,465.0,5500.0,73000.0,"Cycling on TNT Sports @cyclingontnt No lens, n..."
7,https://x.com/komunisnt,komunisn't,@komunisnt,https://x.com/komunisnt/status/181903137304859...,"Aug 1, 2024",NaN,https://x.com/komunisnt/status/181903137304859...,17.0,1300.0,64000.0,komunisn't @komunisnt Japonların dikec yusuf a...
5699,https://x.com/nedenttoldu,@nedenttoldu,Neden TT oldu?,https://x.com/nedenttoldu/status/1820555400174...,"Aug 5, 2024",NaN,https://x.com/nedenttoldu/status/1820555400174...,21.0,270.0,62000.0,@nedenttoldu Neden TT oldu? #Paris2024 https:/...
1202,https://x.com/kultiginmedya,kültigin,@kultiginmedya,https://x.com/kultiginmedya/status/18186842550...,"Jul 31, 2024",🇹🇷,https://x.com/kultiginmedya/status/18186842550...,255.0,2000.0,58000.0,kültigin @kultiginmedya Jandarma Genel Komutan...


📌 Mete Gazoz — En çok görüntülenen ilk 10 tweet


,hesapurl,hesapadi,hesapnickname,tweeturl,time,emoji,alintiurl,yorum,retweet,begeni,tweet
0,https://x.com/Olympics,The Olympic Games,@Olympics,https://x.com/Olympics/status/1421381248400994307,"Jul 31, 2021",NaN,https://x.com/Olympics/status/1421381248400994...,576.0,14000.0,78000.0,The Olympic Games @Olympics Turkey’s first eve...
391,https://x.com/GalatasaraySK,@GalatasaraySK,Galatasaray SK,https://x.com/GalatasaraySK/status/14213799379...,"Jul 31, 2021",🇹🇷,https://x.com/GalatasaraySK/status/14213799379...,187.0,4500.0,52000.0,@GalatasaraySK Galatasaray SK #MeteGazoz https...
475,https://x.com/TMOK_Olimpiyat,TMOK | #TeamTürkiye,@TMOK_Olimpiyat,https://x.com/TMOK_Olimpiyat/status/1421379455...,NaN,🇹🇷,https://x.com/TMOK_Olimpiyat/status/1421379455...,227.0,6400.0,40000.0,TMOK | #TeamTürkiye @TMOK_Olimpiyat #Tokyo2020...
1219,https://x.com/ozanutkuc,Ozan Utku,@ozanutkuc,https://x.com/ozanutkuc/status/142141616635086...,"Jul 31, 2021",NaN,https://x.com/ozanutkuc/status/142141616635086...,116.0,1900.0,35000.0,Ozan Utku @ozanutkuc Gazoz 2009’da minikler ka...
3041,https://x.com/Besiktas,Beşiktaş JK,@Besiktas,https://x.com/Besiktas/status/1421381997012869121,NaN,💪🏻,https://x.com/Besiktas/status/1421381997012869...,63.0,2500.0,25000.0,Beşiktaş JK @Besiktas Ve Altın madalya geliyor...
464,https://x.com/Besiktas,Beşiktaş JK,@Besiktas,https://x.com/Besiktas/status/1421379057397207044,NaN,🇹🇷,https://x.com/Besiktas/status/1421379057397207...,82.0,1700.0,23000.0,"Beşiktaş JK @Besiktas Gazoz Milli okçumuz , #T..."
1706,https://x.com/ntv,NTV,@ntv,https://x.com/ntv/status/1421379992349466627,"Jul 31, 2021",NaN,https://x.com/ntv/status/1421379992349466627/a...,125.0,2900.0,21000.0,NTV @ntv #MeteGazoz https://x.com/hashtag/Mete...
39,https://x.com/bitlikedi,bitlikedi,@bitlikedi,https://x.com/bitlikedi/status/142139790598282...,"Jul 31, 2021",NaN,https://x.com/bitlikedi/status/142139790598282...,51.0,1600.0,20000.0,bitlikedi @bitlikedi Mete 'un 1991'de kurduğu ...
1,https://x.com/Tokyo2020,#Tokyo2020,@Tokyo2020,https://x.com/Tokyo2020/status/142138268643056...,"Jul 31, 2021",🇹🇷,https://x.com/Tokyo2020/status/142138268643056...,125.0,2700.0,18000.0,#Tokyo2020 @Tokyo2020 Mete is the first archer...
465,https://x.com/trtsporyildiz,TRT Spor Yıldız,@trtsporyildiz,https://x.com/trtsporyildiz/status/14213795561...,NaN,🥇,https://x.com/trtsporyildiz/status/14213795561...,90.0,3100.0,11000.0,TRT Spor Yıldız @trtsporyildiz GAZOZ VEEE OLİM...


In [ ]:
import pandas as pd

# Yusuf Dikeç verisi
yusuf_path = "/content/drive/MyDrive/Twitter Verileri/Yusuf_Dikec_Duzenlenmis.csv"
yusuf_df = pd.read_csv(yusuf_path)
print("📌 Yusuf Dikeç verisi — İlk 5 satır")
display(yusuf_df.head())

# Mete Gazoz verisi
mete_path = "/content/drive/MyDrive/Twitter Verileri/Mete_Gazoz_Duzenlenmis.csv"
mete_df = pd.read_csv(mete_path)
print("📌 Mete Gazoz verisi — İlk 5 satır")
display(mete_df.head())


📌 Yusuf Dikeç verisi — İlk 5 satır


,hesapurl,hesapadi,hesapnickname,tweeturl,time,emoji,alintiurl,yorum,retweet,begeni,tweet
0,https://x.com/issf_official,ISSF,@issf_official,https://x.com/issf_official/status/18200369590...,"Aug 4, 2024",👀,https://x.com/issf_official/status/18200369590...,41.0,727.0,3900.0,ISSF @issf_official We found the young Dikec Y...
1,https://x.com/hey_madni,Madni Aghadi,@hey_madni,https://x.com/hey_madni/status/182333168398019...,"Aug 13, 2024",NaN,https://x.com/hey_madni/status/182333168398019...,435.0,3100.0,51000.0,Madni Aghadi @hey_madni AI is getting out of h...
2,https://x.com/ginacostag_,Gina Acosta,@ginacostag_,https://x.com/ginacostag_/status/1822848851931...,"Aug 12, 2024",NaN,NaN,NaN,NaN,NaN,Gina Acosta @ginacostag_
3,https://x.com/NBCOlympics,NBC Olympics & Paralympics,@NBCOlympics,https://x.com/NBCOlympics/status/1822475719668...,"Aug 11, 2024",😤,https://x.com/NBCOlympics/status/1822475719668...,11.0,73.0,625.0,NBC Olympics & Paralympics @NBCOlympics Team G...
4,https://x.com/cyclingontnt,Cycling on TNT Sports,@cyclingontnt,https://x.com/cyclingontnt/status/182235887730...,"Aug 10, 2024",🇬🇧,https://x.com/cyclingontnt/status/182235887730...,61.0,1100.0,15000.0,Cycling on TNT Sports @cyclingontnt Team GB hi...


📌 Mete Gazoz verisi — İlk 5 satır


,hesapurl,hesapadi,hesapnickname,tweeturl,time,emoji,alintiurl,yorum,retweet,begeni,tweet
0,https://x.com/Olympics,The Olympic Games,@Olympics,https://x.com/Olympics/status/1421381248400994307,"Jul 31, 2021",NaN,https://x.com/Olympics/status/1421381248400994...,576.0,14000.0,78000.0,The Olympic Games @Olympics Turkey’s first eve...
1,https://x.com/Tokyo2020,#Tokyo2020,@Tokyo2020,https://x.com/Tokyo2020/status/142138268643056...,"Jul 31, 2021",🇹🇷,https://x.com/Tokyo2020/status/142138268643056...,125.0,2700.0,18000.0,#Tokyo2020 @Tokyo2020 Mete is the first archer...
2,https://x.com/worldarchery,World Archery,@worldarchery,https://x.com/worldarchery/status/142136486317...,"Jul 31, 2021",💪,https://x.com/worldarchery/status/142136486317...,10.0,134.0,822.0,World Archery @worldarchery Mete upsets the wo...
3,https://x.com/TRTWorldNow,TRT World Now,@TRTWorldNow,https://x.com/TRTWorldNow/status/1421381591633...,"Jul 31, 2021",NaN,https://x.com/TRTWorldNow/status/1421381591633...,7.0,101.0,722.0,TRT World Now @TRTWorldNow Mete wins Turkey's ...
4,https://x.com/TRTWorldNow,TRT World Now,@TRTWorldNow,https://x.com/TRTWorldNow/status/1421417857309...,"Jul 31, 2021",NaN,NaN,NaN,NaN,NaN,TRT World Now @TRTWorldNow


In [ ]:
import pandas as pd
import os

# Örnek ana klasör
base_path = "/content/drive/MyDrive/Twitter Verileri"

# İşlenecek klasörler
folders = ["Mete Gazoz", "Yusuf Dikeç"]

# Resmi hesap listesi (gerekirse genişlet)
resmi_hesaplar = ["@genclikspor", "@olimpiyat_tr"]  # federasyon ve bakanlık hesapları
sporcu_hesaplar = ["@metegazoz", "@yusufdikec"]
medya_hesaplar = ["@anadoluajansi", "@trtspor"]

# Tweet içinde resmi ifadeleri kontrol eden fonksiyon
def resmi_tweet(tweet):
    resmi_keywords = ["represent", "team turkey", "official", "delegation", "national"]
    tweet_lower = str(tweet).lower()
    return any(k.lower() in tweet_lower for k in resmi_keywords)

# Hesap tipi atama
def hesap_tipi_guncel(row):
    nickname = row.get("hesapnickname")
    tweet = row.get("tweet", "")

    if nickname in resmi_hesaplar or resmi_tweet(tweet):
        return "RD"
    elif nickname in sporcu_hesaplar:
        return "SK"
    elif nickname in medya_hesaplar:
        return "M"
    elif str(nickname).startswith("@"):
        return "U"
    else:
        return "V"

# Ana kategori atama (geliştirilmiş, hassas)
def ana_kategori(row):
    tweet = str(row.get("tweet", "")).lower()
    hesap_val = row.get("hesap_tipi", "")

    resmi_keywords = ["represent", "team turkey", "official", "delegation", "national"]
    sporcu_keywords = ["culture", "dialog", "interaction", "meeting", "exchange", "ambassador", "visit"]

    if hesap_val == "RD" and any(k in tweet for k in resmi_keywords):
        return "Kategori 1"
    elif hesap_val == "SK" and any(k in tweet for k in sporcu_keywords):
        return "Kategori 2"
    elif hesap_val in ["RD","SK"]:
        return "Hibrit"
    else:
        return "Kategori Dışı"

# Diplomatik etki seviyesi
def etki_seviyesi(retweet):
    try:
        retweet = float(retweet)
    except:
        retweet = 0
    if retweet >= 10000:
        return 3
    elif retweet >= 1000:
        return 2
    else:
        return 1

# İçerik tipi
def icerik_tipi(row):
    tweeturl = row.get("tweeturl")
    alintiurl = row.get("alintiurl")
    if pd.notna(alintiurl):
        return "A"
    elif tweeturl != alintiurl:
        return "RT"
    else:
        return "O"

# İşleme
for folder in folders:
    folder_path = os.path.join(base_path, folder)
    all_dfs = []

    for file in os.listdir(folder_path):
        if file.endswith(".csv"):
            path = os.path.join(folder_path, file)
            try:
                df = pd.read_csv(path)

                # Eksik sütunları doldur
                for col in ["hesapnickname", "tweet", "tweeturl", "alintiurl", "retweet"]:
                    if col not in df.columns:
                        df[col] = None

                # Fonksiyonları uygula
                df["hesap_tipi"] = df.apply(hesap_tipi_guncel, axis=1)
                df["ana_kategori"] = df.apply(ana_kategori, axis=1)
                df["etki_seviyesi"] = df["retweet"].apply(etki_seviyesi)
                df["icerik_tipi"] = df.apply(icerik_tipi, axis=1)

                all_dfs.append(df)
            except Exception as e:
                print(f"Hata: {file} işlenemedi → {e}")

    if all_dfs:
        final_df = pd.concat(all_dfs, ignore_index=True)
        output_file = os.path.join(base_path, f"{folder}_Kodlanmis.csv")
        final_df.to_csv(output_file, index=False, encoding="utf-8-sig")
        print(f"✅ {folder} kaydedildi: {output_file} ({len(final_df)} satır)")



✅ Mete Gazoz kaydedildi: /content/drive/MyDrive/Twitter Verileri/Mete Gazoz_Kodlanmis.csv (10815 satır)
✅ Yusuf Dikeç kaydedildi: /content/drive/MyDrive/Twitter Verileri/Yusuf Dikeç_Kodlanmis.csv (15199 satır)


In [ ]:
import pandas as pd
import os

# Dosya yolları
base_path = "/content/drive/MyDrive/Twitter Verileri"
yusuf_file = os.path.join(base_path, "Yusuf_Dikec_Duzenlenmis.csv")
mete_file = os.path.join(base_path, "Mete_Gazoz_Duzenlenmis.csv")
output_file = os.path.join(base_path, "Twitter_MurrayPigman_Coded.csv")

# ------------------------------
# Hesap tipini tespit et
# ------------------------------
def detect_account_type(nickname):
    nickname = str(nickname).lower()

    official_accounts = ['genclikspor', 'olimpiyat_tr', 'tcbestepe', 'tcsosyalmedya']
    athlete_accounts = ['yusufdikec', 'metegazoz']
    media_keywords = ['haber', 'spor', 'news', 'sport', 'gazete', 'tv']

    if any(acc in nickname for acc in official_accounts):
        return 'RD'
    elif any(acc in nickname for acc in athlete_accounts):
        return 'SK'
    elif any(keyword in nickname for keyword in media_keywords):
        return 'M'
    else:
        return 'V'

# ------------------------------
# Murray-Pigman ana kategori tespiti
# ------------------------------
CATEGORY1_KEYWORDS = [
    'türkiye\'yi temsil', 'milli sporcumuz', 'bayrağımızı',
    'ülkemizin gururu', 'milli başarı', 'representing turkey',
    'national pride', 'our athlete',
    'archery', 'arrow', 'recurve', 'compound bow', 'archer',
    'shooting', 'rifle', 'pistol', 'marksman', 'issf', 'target'
]

CATEGORY2_KEYWORDS = [
    'thank you', 'teşekkür', 'kültürümüz', 'istanbul', 'welcome',
    'friendship', 'dostluk', 'peace', 'barış', 'together'
]

NON_DIPLOMATIC = [
    'antrenman', 'training', 'technique', 'teknik', 'skor', 'score',
    'performans', 'performance', 'metre', 'meter'
]

def categorize_murray_pigman(tweet_text, account_type):
    text_lower = str(tweet_text).lower()
    cat1_score = sum(1 for kw in CATEGORY1_KEYWORDS if kw in text_lower)
    cat2_score = sum(1 for kw in CATEGORY2_KEYWORDS if kw in text_lower)
    non_dip_score = sum(1 for kw in NON_DIPLOMATIC if kw in text_lower)

    if account_type == 'RD' and cat1_score > 0:
        return 'K1'
    elif account_type == 'SK' and cat2_score > 0:
        return 'K2'
    elif cat1_score > 0 and cat2_score > 0:
        return 'H'  # Hibrit
    elif non_dip_score > cat1_score + cat2_score:
        return 'D'  # Diplomasi dışı
    elif cat1_score > cat2_score:
        return 'K1'
    elif cat2_score > cat1_score:
        return 'K2'
    else:
        return 'D'

# ------------------------------
# Diplomatik etki seviyesi
# ------------------------------
def calculate_diplomatic_impact(retweet, like, reply=0, quote=0):
    # Ensure only numeric types are summed
    numeric_interactions = [x for x in [retweet, like, reply, quote] if isinstance(x, (int, float)) and pd.notna(x)]
    total = sum(numeric_interactions)
    if total >= 10000:
        return 3
    elif total >= 1000:
        return 2
    else:
        return 1

# ------------------------------
# İçerik tipi tespiti
# ------------------------------
def detect_content_type(tweet_text):
    text = str(tweet_text)
    if text.startswith('RT @'):
        return 'RT'
    elif text.startswith('@'):
        return 'Y'
    elif 'http' in text and '"' in text: # Assuming a quoted tweet contains a URL and quotes
        return 'A'
    else:
        return 'O'

# ------------------------------
# Ana işleme fonksiyonu
# ------------------------------
def process_and_code_dataframe(df, athlete_name):
    df_coded = df.copy()

    df_coded['Hesap_Tipi'] = df_coded['hesapnickname'].apply(detect_account_type)
    df_coded['Ana_Kategori'] = df_coded.apply(lambda r: categorize_murray_pigman(r['tweet'], r['Hesap_Tipi']), axis=1)
    df_coded['Diplomatik_Etki'] = df_coded.apply(lambda r: calculate_diplomatic_impact(r['retweet'], r['begeni'], r.get('yorum',0), r.get('alintiurl',0)), axis=1)
    df_coded['İçerik_Tipi'] = df_coded['tweet'].apply(detect_content_type)
    df_coded['Sporcu'] = 'YD' if athlete_name.lower().startswith('yusuf') else 'MG'

    # Sum only numeric interaction columns
    df_coded['Etkilesim_Sayisi'] = df_coded[['retweet','begeni','yorum']].sum(axis=1, skipna=True)

    return df_coded

# ------------------------------
# CSV dosyalarını oku ve kodla
# ------------------------------
yusuf_df = pd.read_csv(yusuf_file)
mete_df = pd.read_csv(mete_file)

yusuf_coded = process_and_code_dataframe(yusuf_df, 'Yusuf_Dikec')
mete_coded = process_and_code_dataframe(mete_df, 'Mete_Gazoz')

# Birleştir ve kaydet
all_coded = pd.concat([yusuf_coded, mete_coded], ignore_index=True)
all_coded.to_csv(output_file, index=False, encoding="utf-8-sig")

print("✅ Kodlama tamamlandı, dosya kaydedildi:", output_file)
print("Ana kategoriler dağılımı:\n", all_coded['Ana_Kategori'].value_counts())

✅ Kodlama tamamlandı, dosya kaydedildi: /content/drive/MyDrive/Twitter Verileri/Twitter_MurrayPigman_Coded.csv
Ana kategoriler dağılımı:
 Ana_Kategori
D     8743
K1    1030
K2     172
H       30
Name: count, dtype: int64


OK

In [ ]:
# Hesap tiplerini tespit eden fonksiyon
def detect_account_type(row):
    username = str(row.get('hesapnickname', '')).lower().replace('@','')
    description = str(row.get('hesapadi', '')).lower()
    verified = row.get('verified', False)  # Eğer yoksa False varsay

    # Sporcu kişisel hesaplar (SK)
    if username in ['metegazoz', 'yusufdikec']:
        return 'SK'

    # Resmi devlet hesapları (RD)
    rd_keywords = ['genclikspor', 'olimpiyat_tr', 'federasyon', 'bakanligi', 'tc']
    if any(kw in username for kw in rd_keywords):
        return 'RD'

    # Medya hesapları (M)
    media_keywords = ['haber', 'ajansi', 'gazete', 'spor', 'news', 'tv']
    if any(kw in username for kw in media_keywords):
        return 'M'

    # Uluslararası hesaplar (U)
    intl_keywords = ['olympics', 'worldarchery', 'issf', 'international', 'ioc']
    if any(kw in username for kw in intl_keywords) or "international" in description:
        return 'U'
    if verified and username not in ['metegazoz', 'yusufdikec']:
        return 'U'

    # Varsayılan: Vatandaş (V)
    return 'V'

# all_coded veri setine uygula
all_coded['Hesap_Tipi'] = all_coded.apply(detect_account_type, axis=1)

# Hesap tiplerini tablo olarak çıkar
hesap_tipi_tablosu = all_coded['Hesap_Tipi'].value_counts().reset_index()
hesap_tipi_tablosu.columns = ['Hesap_Tipi', 'Sayı']

# Göster
print("📊 Hesap Tipleri Dağılımı:")
print(hesap_tipi_tablosu)






📊 Hesap Tipleri Dağılımı:
  Hesap_Tipi  Sayı
0          V  8905
1          M   948
2         RD    90
3          U    28
4         SK     4


OK

In [ ]:
import pandas as pd
import numpy as np # Import numpy

def group_by_diplomatic_impact(df):
    """
    Retweet sayılarına göre diplomatik etki seviyesini gruplandırır
    """
    conditions = [
        (df['retweet'] < 1000),
        (df['retweet'] >= 1000) & (df['retweet'] < 10000),
        (df['retweet'] >= 10000)
    ]
    choices = [1, 2, 3]  # Düşük, Orta, Yüksek

    df['Diplomatik_Etki_Seviyesi'] = np.select(conditions, choices, default=1)

    # Gruplama tablosu
    summary = df.groupby('Diplomatik_Etki_Seviyesi').size().reset_index(name='Tweet_Sayısı')
    return df, summary

# Örnek kullanım (hem Yusuf hem Mete için):
# Assuming 'yusuf_coded' and 'mete_coded' DataFrames are available from previous steps
yusuf_with_impact, yusuf_summary = group_by_diplomatic_impact(yusuf_coded)
mete_with_impact, mete_summary = group_by_diplomatic_impact(mete_coded)

print("📊 Yusuf Dikeç - Diplomatik Etki Özeti")
print(yusuf_summary)

print("\n📊 Mete Gazoz - Diplomatik Etki Özeti")
print(mete_summary)

📊 Yusuf Dikeç - Diplomatik Etki Özeti
   Diplomatik_Etki_Seviyesi  Tweet_Sayısı
0                         1          6050
1                         2           570
2                         3           156

📊 Mete Gazoz - Diplomatik Etki Özeti
   Diplomatik_Etki_Seviyesi  Tweet_Sayısı
0                         1          3179
1                         2            18
2                         3             2


OK

In [ ]:
import pandas as pd

# Dosya yolları
yusuf_path = "/content/drive/MyDrive/Twitter Verileri/Yusuf_Dikec_Duzenlenmis.csv"
mete_path = "/content/drive/MyDrive/Twitter Verileri/Mete_Gazoz_Duzenlenmis.csv"

# Dosyaları oku
yusuf_df = pd.read_csv(yusuf_path)
mete_df = pd.read_csv(mete_path)

def detect_content_types(df):
    df = df.copy()

    # Varsayılan = Orijinal
    df["icerik_tipi"] = "O"

    # 1. Mention / reply
    df.loc[df["tweet"].str.startswith("@", na=False), "icerik_tipi"] = "Y"

    # 2. Quote tweet = alintiurl farklı ve dolu ise
    df.loc[(df["alintiurl"].notna()) & (df["alintiurl"] != "") & (df["alintiurl"] != df["tweeturl"]), "icerik_tipi"] = "A"

    # 3. RT = aynı tweeturl birden fazla kez varsa, ilk dışındakiler RT
    duplicate_urls = df["tweeturl"].value_counts()
    duplicate_urls = duplicate_urls[duplicate_urls > 1].index

    for url in duplicate_urls:
        indices = df.index[df["tweeturl"] == url].tolist()
        if len(indices) > 1:
            # ilkini orijinal bırak, diğerlerini RT yap
            df.loc[indices[1:], "icerik_tipi"] = "RT"

    return df

# Uygula
yusuf_df = detect_content_types(yusuf_df)
mete_df = detect_content_types(mete_df)

# Aynı dosyalara kaydet
yusuf_df.to_csv(yusuf_path, index=False, encoding="utf-8-sig")
mete_df.to_csv(mete_path, index=False, encoding="utf-8-sig")

# Kontrol için özet tablo
print("📊 Yusuf Dikeç İçerik Tipleri:")
print(yusuf_df["icerik_tipi"].value_counts())

print("\n📊 Mete Gazoz İçerik Tipleri:")
print(mete_df["icerik_tipi"].value_counts())



📊 Yusuf Dikeç İçerik Tipleri:
icerik_tipi
A    5618
Y     624
O     534
Name: count, dtype: int64

📊 Mete Gazoz İçerik Tipleri:
icerik_tipi
A    2731
O     282
Y     186
Name: count, dtype: int64


OK

In [ ]:
import pandas as pd

# Dosya yolları
yusuf_path = "/content/drive/MyDrive/Twitter Verileri/Yusuf_Dikec_Duzenlenmis.csv"
mete_path = "/content/drive/MyDrive/Twitter Verileri/Mete_Gazoz_Duzenlenmis.csv"

# Dosyaları oku
yusuf_df = pd.read_csv(yusuf_path)
mete_df = pd.read_csv(mete_path)

# Fonksiyon: Dil tespiti (basitleştirilmiş)
def detect_language(text):
    try:
        text_lower = str(text).lower()
        if any(word in text_lower for word in ['the', 'and', 'is', 'you', 'thank', 'congrat', 'gold', 'medal',
            'win', 'proud', 'archery', 'shooter', 'olympics']):
            return 'EN'
        elif any(word in text_lower for word in ['ve', 'bir', 'ile', 'da', 'çok', 'teşekkür', 'gurur', 'altın',
            'madalya', 'sporcu', 'başarı', 'okçuluk', 'atıcı', 'olimpiyat']):
            return 'TR'
        else:
            return 'DİĞ'
    except:
        return 'DİĞ'

# Fonksiyon: İçerik tipi tespiti
def detect_content_types(df):
    df = df.copy()
    df["icerik_tipi"] = "O"  # varsayılan Orijinal

    # Mention / reply
    df.loc[df["tweet"].str.startswith("@", na=False), "icerik_tipi"] = "Y"

    # Quote tweet (alıntı)
    df.loc[(df["alintiurl"].notna()) & (df["tweeturl"] != df["alintiurl"]), "icerik_tipi"] = "A"

    # Retweet = aynı tweeturl birden fazla kez varsa, ilk dışındakiler RT
    duplicate_urls = df["tweeturl"].value_counts()
    duplicate_urls = duplicate_urls[duplicate_urls > 1].index
    for url in duplicate_urls:
        indices = df.index[df["tweeturl"] == url].tolist()
        if len(indices) > 1:
            df.loc[indices[1:], "icerik_tipi"] = "RT"

    return df

# Fonksiyon: Diplomatik etki seviyesi
def calculate_diplomatic_impact(row):
    total_engagement = sum([
        row.get("retweet", 0) if not pd.isna(row.get("retweet")) else 0,
        row.get("begeni", 0) if not pd.isna(row.get("begeni")) else 0,
        row.get("yorum", 0) if not pd.isna(row.get("yorum")) else 0
    ])
    if total_engagement >= 10000:
        return 3  # Yüksek
    elif total_engagement >= 1000:
        return 2  # Orta
    else:
        return 1  # Düşük

# Fonksiyon: Hesap tipi tespiti
def detect_account_type(row, athlete_name):
    username = str(row.get("hesapnickname", "")).lower()
    official_accounts = ['genclikspor', 'olimpiyat_tr', 'tcbestepe', 'tcsosyalmedya']
    athlete_accounts = ['yusufdikec', 'metegazoz']
    media_keywords = ['haber', 'spor', 'news', 'sport', 'gazete', 'tv']

    if any(acc in username for acc in official_accounts):
        return 'RD'
    elif any(acc in username for acc in athlete_accounts):
        return 'SK'
    elif any(keyword in username for keyword in media_keywords):
        return 'M'
    elif row.get("verified", False):
        return 'U'
    else:
        return 'V'

# Fonksiyon: Ana kategori tespiti (K1/K2/H/D)
def categorize_murray_pigman(row):
    text_lower = str(row.get("tweet", "")).lower()
    account_type = row.get("Hesap_Tipi", "V")

    cat1_indicators = [
    'türkiye\'yi temsil', 'milli sporcumuz', 'bayrağımızı', 'ülkemizin gururu',
    'milli başarı', 'gururumuz', 'altın madalya', 'şampiyon', 'kahraman',
    'okçuluk', 'atış', 'atıcı', 'shooting', 'archery',
    'representing turkey', 'national pride', 'proud of turkey',
    'champion', 'gold medal', 'hero'
]


    cat2_indicators = [
    'teşekkür', 'kültürümüz', 'hoş geldin', 'misafirperver', 'kardeşlik',
    'dostluk', 'barış', 'birlik', 'dayanışma', 'selam',
    'thank you', 'friendship', 'peace', 'together', 'solidarity',
    'welcome', 'respect', 'congratulations'
]


    cat1_score = sum(1 for k in cat1_indicators if k in text_lower)
    cat2_score = sum(1 for k in cat2_indicators if k in text_lower)

    if account_type == 'RD' and cat1_score > 0:
        return 'K1'
    elif account_type == 'SK' and cat2_score > 0:
        return 'K2'
    elif cat1_score > 0 and cat2_score > 0:
        return 'H'
    elif cat1_score == 0 and cat2_score == 0:
        return 'D'
    elif cat1_score > cat2_score:
        return 'K1'
    else:
        return 'K2'

# Fonksiyon: İşleme
def process_dataset(df, athlete_name):
    df = df.copy()
    df["Sporcu"] = "YD" if athlete_name == "Yusuf_Dikec" else "MG"
    df = detect_content_types(df)
    df["Hesap_Tipi"] = df.apply(lambda row: detect_account_type(row, athlete_name), axis=1)
    df["Diplomatik_Etki"] = df.apply(calculate_diplomatic_impact, axis=1)
    df["Ana_Kategori"] = df.apply(categorize_murray_pigman, axis=1)
    df["Etkileşim_Sayısı"] = df[["retweet","begeni","yorum"]].sum(axis=1, skipna=True)
    df["Dil"] = df["tweet"].apply(detect_language)
    df["Tweet_ID"] = [f"{athlete_name}_{i+1}" for i in range(len(df))]
    return df[
        ["Tweet_ID","Sporcu","Ana_Kategori","Hesap_Tipi",
         "Diplomatik_Etki","icerik_tipi","Etkileşim_Sayısı","Dil","tweet"]
    ].rename(columns={"tweet":"Tweet_Metni"})

# Uygula
yusuf_coded = process_dataset(yusuf_df, "Yusuf_Dikec")
mete_coded = process_dataset(mete_df, "Mete_Gazoz")

# Kaydet
yusuf_coded.to_csv(yusuf_path, index=False, encoding="utf-8-sig")
mete_coded.to_csv(mete_path, index=False, encoding="utf-8-sig")

# Özet
print("📊 Yusuf Dikeç:")
print(yusuf_coded["Ana_Kategori"].value_counts())
print(yusuf_coded["icerik_tipi"].value_counts())

print("\n📊 Mete Gazoz:")
print(mete_coded["Ana_Kategori"].value_counts())
print(mete_coded["icerik_tipi"].value_counts())


📊 Yusuf Dikeç:
Ana_Kategori
D     5624
K1     971
K2     153
H       28
Name: count, dtype: int64
icerik_tipi
A    5618
Y     624
O     534
Name: count, dtype: int64

📊 Mete Gazoz:
Ana_Kategori
D     1947
K1    1083
H       87
K2      82
Name: count, dtype: int64
icerik_tipi
A    2731
O     282
Y     186
Name: count, dtype: int64


OK

In [ ]:
def print_tweets_with_text(df):
    for _, row in df.iterrows():
        print(f"Tweet_ID: {row['Tweet_ID']}")
        print(f"Sporcu: {row['Sporcu']}")
        print(f"Ana_Kategori: {row['Ana_Kategori']}")
        print(f"Hesap_Tipi: {row['Hesap_Tipi']}")
        print(f"Diplomatik_Etki: {row['Diplomatik_Etki']}")
        print(f"İçerik_Tipi: {row['icerik_tipi']}")
        print(f"Etkileşim_Sayısı: {row['Etkileşim_Sayısı']}")
        print(f"Dil: {row['Dil']}")
        print(f"Tweet Metni: \"{row['Tweet_Metni']}\"")
        print("-" * 50)

# Örnek kullanım
print("📊 Yusuf Dikeç Tweetleri:")
print_tweets_with_text(yusuf_coded)

print("\n📊 Mete Gazoz Tweetleri:")
print_tweets_with_text(mete_coded)


Görüntülenen çıkış son 5000 satıra kısaltıldı.
--------------------------------------------------
Tweet_ID: Mete_Gazoz_2717
Sporcu: MG
Ana_Kategori: D
Hesap_Tipi: V
Diplomatik_Etki: 1
İçerik_Tipi: A
Etkileşim_Sayısı: 0.0
Dil: TR
Tweet Metni: "@agahpamukov Orhan Pamukov Okan Can Yantır'ın tweet'leri çok komik ya tamam kardeşim sen yarattın Gazoz Mete Mete 'u tamam daha fazla paylaşım yapma ikna olduk en büyük emek senin, Gazoz"
--------------------------------------------------
Tweet_ID: Mete_Gazoz_2718
Sporcu: MG
Ana_Kategori: D
Hesap_Tipi: V
Diplomatik_Etki: 1
İçerik_Tipi: A
Etkileşim_Sayısı: 5.0
Dil: TR
Tweet Metni: "@IbrahimKOLDAMCA 𝓘̇𝓫𝓻𝓪𝓱𝓲𝓶 𝓚𝓞𝓛𝓓𝓐𝓜𝓒𝓐 OLİMPİYAT ŞAMPİYONU GAZOZ ALTIN MADALYA METE ’UN FİNALİ GELDİ . ZOR GÜNDE BİZLERİ MUTLU ETTİN KARDEŞİM"
--------------------------------------------------
Tweet_ID: Mete_Gazoz_2719
Sporcu: MG
Ana_Kategori: H
Hesap_Tipi: V
Diplomatik_Etki: 1
İçerik_Tipi: A
Etkileşim_Sayısı: 2.0
Dil: TR
Tweet Metni: "@ahmetaykol Ahmet Aykol Tokyo 2020 Oli

OK

In [ ]:
import pandas as pd

# Dosya yolları
yusuf_path = "/content/drive/MyDrive/Twitter Verileri/Yusuf_Dikec_Duzenlenmis.csv"
mete_path = "/content/drive/MyDrive/Twitter Verileri/Mete_Gazoz_Duzenlenmis.csv"

# Dosyaları oku
yusuf_df = pd.read_csv(yusuf_path)
mete_df = pd.read_csv(mete_path)

def calculate_summary(df):
    total_tweets = len(df)
    cat1_pct = round(len(df[df['Ana_Kategori'] == 'K1']) / total_tweets * 100, 1)
    cat2_pct = round(len(df[df['Ana_Kategori'] == 'K2']) / total_tweets * 100, 1)
    avg_diplomatic = round(df['Diplomatik_Etki'].mean(), 1)

    # En viral etkileşim: sadece yorum + retweet + beğeni toplamı
    if {'yorum', 'retweet', 'begeni'}.issubset(df.columns):
        max_engagement = (df['yorum'].fillna(0) + df['retweet'].fillna(0) + df['begeni'].fillna(0)).max()
    else:
        max_engagement = df['Etkileşim_Sayısı'].max()

    dominant_account = df['Hesap_Tipi'].mode()[0] if not df['Hesap_Tipi'].mode().empty else None

    return {
        "Toplam Tweet Sayısı": total_tweets,
        "Kategori 1 (Araç) %": f"{cat1_pct}%",
        "Kategori 2 (Diplomatı) %": f"{cat2_pct}%",
        "Ortalama Diplomatik Etki": avg_diplomatic,
        "En Viral Tweet Etkileşimi": max_engagement,
        "Dominant Hesap Tipi": dominant_account
    }

# Özetleri hesapla
yusuf_summary = calculate_summary(yusuf_df)
mete_summary = calculate_summary(mete_df)

# Tek tablo olarak birleştir
summary_df = pd.DataFrame([yusuf_summary, mete_summary], index=["Yusuf Dikeç", "Mete Gazoz"])
summary_df = summary_df.reset_index().rename(columns={"index": "Ölçüm"})

# Yazdır
print(summary_df.to_string(index=False))


      Ölçüm  Toplam Tweet Sayısı Kategori 1 (Araç) % Kategori 2 (Diplomatı) %  Ortalama Diplomatik Etki  En Viral Tweet Etkileşimi Dominant Hesap Tipi
Yusuf Dikeç                 6776               14.3%                     2.3%                       1.2                   319000.0                   V
 Mete Gazoz                 3199               33.9%                     2.6%                       1.0                    92576.0                   V


OK


In [ ]:
!pip install langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 18.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=102b930edf4635daa1568ec11970c497777373089c8bb3fd49977c5baa696acb
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


OK

In [ ]:
import pandas as pd
import json

# ---------- KODLANDIRMA FONKSİYONLARI ----------

def detect_account_type(username, verified):
    """
    Hesap tipini tespit eder: RD/SK/M/V/U
    """
    username = str(username).lower().replace('@', '')

    # Sporcu kişisel hesaplar
    if username in ['metegazoz', 'yusufdikec']:
        return 'SK'

    # Resmi devlet hesapları
    rd_keywords = ['genclikspor', 'olimpiyat_tr', 'federasyon', 'bakanligi', 'tc']
    if any(kw in username for kw in rd_keywords):
        return 'RD'

    # Medya hesapları
    media_keywords = ['haber', 'ajansi', 'gazete', 'spor', 'news', 'tv']
    if any(kw in username for kw in media_keywords):
        return 'M'

    # Uluslararası hesaplar
    intl_keywords = ['olympics', 'worldarchery', 'issf', 'international', 'ioc']
    if any(kw in username for kw in intl_keywords) or verified:
        return 'U'

    # Varsayılan: Vatandaş
    return 'V'


def categorize_murray_pigman(tweet_text, account_type):
    """
    Murray-Pigman kategorilerine göre sınıflandırır: K1/K2/H/D
    """
    text_lower = str(tweet_text).lower()

    category1_indicators = [
        'türkiye\'yi temsil', 'milli sporcumuz', 'bayrağımızı', 'ülkemizin gururu',
        'milli başarı', 'representing turkey', 'national pride', 'our athlete'
    ]

    category2_indicators = [
        'thank you', 'teşekkür', 'kültürümüz', 'istanbul', 'welcome',
        'friendship', 'dostluk', 'peace', 'barış', 'together'
    ]

    non_diplomatic = [
        'antrenman', 'training', 'technique', 'teknik', 'skor', 'score',
        'performans', 'performance', 'metre', 'meter'
    ]

    cat1_score = sum(1 for indicator in category1_indicators if indicator in text_lower)
    cat2_score = sum(1 for indicator in category2_indicators if indicator in text_lower)
    non_dip_score = sum(1 for indicator in non_diplomatic if indicator in text_lower)

    if account_type == 'RD' and cat1_score > 0:
        return 'K1'
    elif account_type == 'SK' and cat2_score > 0:
        return 'K2'
    elif cat1_score > 0 and cat2_score > 0:
        return 'H'
    elif non_dip_score > cat1_score + cat2_score:
        return 'D'
    elif cat1_score > cat2_score:
        return 'K1'
    elif cat2_score > cat1_score:
        return 'K2'
    else:
        return 'D'


def calculate_diplomatic_impact(retweet, like, reply, quote):
    """
    Diplomatik etki seviyesini hesaplar (1-3 arası)
    """
    total_engagement = retweet + like + reply + quote

    if total_engagement >= 10000:
        return 3
    elif total_engagement >= 1000:
        return 2
    else:
        return 1


def detect_content_type(tweet_text):
    """
    İçerik tipini tespit eder: O/RT/Y/A
    """
    text = str(tweet_text)
    if text.startswith('RT @'):
        return 'RT'
    elif text.startswith('@'):
        return 'Y'
    elif 'twitter.com' in text and '"' in text:
        return 'A'
    else:
        return 'O'


# ---------- ANA KODLANDIRMA İŞLEMI ----------

def process_and_code_tweets(df, athlete_name):
    """
    Tüm tweet'leri kodlar ve Murray-Pigman kategorilerine ayırır
    """
    coded_tweets = []

    for idx, row in df.iterrows():
        # Hesap tipi
        account_type = detect_account_type(row.get('hesapnickname', ''), row.get('verified', False))

        # Diplomatik etki
        diplomatic_impact = calculate_diplomatic_impact(
            row.get('retweet', 0),
            row.get('begeni', 0),
            row.get('yorum', 0),
            row.get('alinti', 0)  # eğer alıntı sayısı varsa
        )

        # Murray-Pigman kategori
        category = categorize_murray_pigman(row.get('Tweet_Metni', ''), account_type)

        # İçerik tipi
        content_type = detect_content_type(row.get('Tweet_Metni', ''))

        # Toplam etkileşim
        total_engagement = row.get('retweet',0) + row.get('begeni',0) + row.get('yorum',0)

        coded_tweet = {
            'Tweet_ID': f"{athlete_name}_{idx:04d}",
            'Sporcu': 'YD' if athlete_name.lower() == 'yusuf_dikec' else 'MG',
            'Ana_Kategori': category,
            'Hesap_Tipi': account_type,
            'Diplomatik_Etki': diplomatic_impact,
            'icerik_tipi': content_type,
            'Etkileşim_Sayısı': total_engagement,
            'Dil': row.get('Dil', 'TR'),
            'Tweet_Metni': row.get('Tweet_Metni', '')[:100] + '...' if len(str(row.get('Tweet_Metni',''))) > 100 else row.get('Tweet_Metni','')
        }

        coded_tweets.append(coded_tweet)

    return pd.DataFrame(coded_tweets)


# ---------- RAPORLAMA ----------

def generate_murray_pigman_report(coded_df):
    """
    Murray-Pigman analiz raporu oluşturur
    """
    report = {}
    for athlete in ['YD', 'MG']:
        athlete_data = coded_df[coded_df['Sporcu'] == athlete].copy()

        report[athlete] = {
            'Toplam_Tweet': len(athlete_data),
            'K1_Yüzde': len(athlete_data[athlete_data['Ana_Kategori'] == 'K1']) / len(athlete_data) * 100 if len(athlete_data)>0 else 0,
            'K2_Yüzde': len(athlete_data[athlete_data['Ana_Kategori'] == 'K2']) / len(athlete_data) * 100 if len(athlete_data)>0 else 0,
            'Hibrit_Yüzde': len(athlete_data[athlete_data['Ana_Kategori'] == 'H']) / len(athlete_data) * 100 if len(athlete_data)>0 else 0,
            'Ortalama_Diplomatik_Etki': athlete_data['Diplomatik_Etki'].mean() if len(athlete_data)>0 else 0,
            'En_Viral_Tweet': athlete_data.loc[athlete_data['Etkileşim_Sayısı'].idxmax(), 'Tweet_Metni'] if len(athlete_data)>0 else '',
            'Dominant_Kategori': athlete_data['Ana_Kategori'].mode()[0] if len(athlete_data)>0 else ''
        }

    return report





OK

In [ ]:
print(yusuf_df.columns)


Index(['Tweet_ID', 'Sporcu', 'Ana_Kategori', 'Hesap_Tipi', 'Diplomatik_Etki',
       'icerik_tipi', 'Etkileşim_Sayısı', 'Dil', 'Tweet_Metni'],
      dtype='object')


OK

In [ ]:
all_coded_tweets = pd.concat([yusuf_coded, mete_coded], ignore_index=True)


OK

In [ ]:
report = generate_murray_pigman_report(all_coded_tweets)
import json
print(json.dumps(report, indent=2, ensure_ascii=False))

{
  "YD": {
    "Toplam_Tweet": 6776,
    "K1_Yüzde": 14.329988193624557,
    "K2_Yüzde": 2.257969303423849,
    "Hibrit_Yüzde": 0.4132231404958678,
    "Ortalama_Diplomatik_Etki": 1.2352420306965761,
    "En_Viral_Tweet": "Yusuf Dikec @yusufdikec Hi Elon, do you think future robots can win medals at the Olympics with their hands in their pockets? How about discussing this in Istanbul, the cultural capital that unites continents?",
    "Dominant_Kategori": "D"
  },
  "MG": {
    "Toplam_Tweet": 3199,
    "K1_Yüzde": 33.85432947796186,
    "K2_Yüzde": 2.5633010315723666,
    "Hibrit_Yüzde": 2.719599874960925,
    "Ortalama_Diplomatik_Etki": 1.0412628946545797,
    "En_Viral_Tweet": "The Olympic Games @Olympics Turkey’s first ever Olympic medal in Gazoz #Archery https://x.com/hashtag/Archery?src=hashtag_click Mete of",
    "Dominant_Kategori": "D"
  }
}


Ok


In [ ]:
import pandas as pd

# 1️⃣ Kodlanmış tweet verilerini Excel’e kaydet
all_coded_tweets.to_excel("/content/drive/MyDrive/Twitter Verileri/Coded_Tweets.xlsx",
                          index=False, sheet_name="Coded_Tweets")

# 2️⃣ Analiz raporunu DataFrame’e çevirip ayrı bir sheet’e kaydet
report_df_list = []

for athlete, data in report.items():
    temp = pd.DataFrame([data])
    temp.insert(0, "Sporcu", athlete)
    report_df_list.append(temp)

report_df = pd.concat(report_df_list, ignore_index=True)

with pd.ExcelWriter("/content/drive/MyDrive/Twitter Verileri/Coded_Tweets.xlsx",
                    mode='a', engine='openpyxl') as writer:
    report_df.to_excel(writer, index=False, sheet_name="Analysis_Report")

print("✅ Excel dosyası oluşturuldu ve hem kodlanmış tweetler hem rapor kaydedildi.")


✅ Excel dosyası oluşturuldu ve hem kodlanmış tweetler hem rapor kaydedildi.
